# Florida Save Our Homes benefit by county

This notebook loads `data/counties.csv` and `data/distribution_2026.csv` and reproduces the headline figures in the README. Each row of `counties.csv` is one county in one roll year.

In [1]:
import pandas as pd

df = pd.read_csv('../data/counties.csv')
dist = pd.read_csv('../data/distribution_2026.csv')
df.shape

(335, 12)

## The statewide differential by year

The 2026 county figures sum exactly to line 12 of DOR's 2026 preliminary DR-489V recapitulation, $815,135,591,712.

In [2]:
by_year = df.groupby('year')['soh_differential_total'].sum()
assert by_year[2026] == 815_135_591_712
(by_year / 1e9).round(1)

year
2022    679.6
2023    893.2
2024    929.1
2025    890.9
2026    815.1
Name: soh_differential_total, dtype: float64

## Estimated tax the cap saves the average homestead, 2026

An estimate: the average differential times the county's 2025 Total Millage Rate, divided by 1,000. The average is recomputed from the NAL totals rather than the rounded column, which is how the dataset computes it.

In [3]:
m25 = df[df.year == 2025].set_index('county')['avg_total_millage']
y26 = df[df.year == 2026].set_index('county')
avg = (y26['homestead_just_value_total'] - y26['homestead_assessed_value_total']) / y26['homestead_parcels']
check = (avg * m25 / 1000).round()
assert (check == y26['est_avg_annual_tax_saved']).all()
y26[['homestead_parcels', 'avg_soh_differential_per_homestead', 'est_avg_annual_tax_saved']].sort_values('est_avg_annual_tax_saved', ascending=False).head(10)

,homestead_parcels,avg_soh_differential_per_homestead,est_avg_annual_tax_saved
county,,,
Miami-Dade,451377.0,294035.0,5436.0
Palm Beach,366108.0,287327.0,5026.0
Broward,419468.0,230895.0,4586.0
Monroe,15702.0,475825.0,3919.0
Martin,48966.0,234670.0,3769.0
Pinellas,250466.0,162443.0,3004.0
Indian River,51024.0,201725.0,2858.0
Nassau,31913.0,171243.0,2749.0
Orange,255817.0,153863.0,2679.0


## Homesteads above the $500,000 portability limit

Portability carries at most $500,000 of differential to a new Florida homestead. The part above it is lost on a move.

In [4]:
d = dist.set_index('county')
above = d['differential_above_500k_total'] * m25 / 1000
print(f"{d['parcels_differential_over_500k'].sum():,} homesteads, "
      f"${d['differential_above_500k_total'].sum() / 1e9:.1f} billion above the limit, "
      f"about ${above.sum() / 1e9:.2f} billion a year in tax at 2025 rates")

187,319 homesteads, $122.7 billion above the limit, about $2.06 billion a year in tax at 2025 rates
